# COMP 472 — Brain Tumor MRI Classification
## ResNet-34
### Note
Only **Cell 3** needs your input. Replace the `None` placeholders with your DataLoaders for each dataset. Everything else runs automatically.

## 1. Setup

In [11]:
!pip install torchmetrics grad-cam --quiet

In [12]:
import copy, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchmetrics import Accuracy, Precision, Recall, F1Score
from sklearn.metrics import classification_report, confusion_matrix
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from google.colab import drive

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

drive.mount('/content/drive', force_remount=True)
OUTPUT_DIR = Path('/content/drive/MyDrive/COMP472/outputs/resnet34')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda
Mounted at /content/drive


## 2. Hyperparameters

In [13]:
BATCH_SIZE    = 32
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
IMG_SIZE      = 224

## 3. Data (TO FILL)

Replace each `None` with your DataLoader.  
Images must be tensors of shape `(batch, 3, 224, 224)`, normalized with:
```
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

In [14]:
# ── Dataset 1 : 3,264 images — 4 classes ──────────────────────────
train_loader_ds1 = None   # ← your DataLoader
val_loader_ds1   = None   # ← your DataLoader
test_loader_ds1  = None   # ← your DataLoader
class_names_ds1  = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# ── Dataset 2 : 7,023 images — 4 classes ──────────────────────────
train_loader_ds2 = None   # ← your DataLoader
val_loader_ds2   = None   # ← your DataLoader
test_loader_ds2  = None   # ← your DataLoader
class_names_ds2  = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# ── Dataset 3 : 4,478 images — 44 classes ─────────────────────────
train_loader_ds3 = None   # ← your DataLoader
val_loader_ds3   = None   # ← your DataLoader
test_loader_ds3  = None   # ← your DataLoader
class_names_ds3  = None   # ← list of 44 class name strings

DATASETS = [
    {
        'name': 'Dataset 1 — Brain Tumor Classification (MRI)',
        'num_classes': 4, 'class_names': class_names_ds1,
        'train_loader': train_loader_ds1, 'val_loader': val_loader_ds1, 'test_loader': test_loader_ds1,
    },
    {
        'name': 'Dataset 2 — Brain Tumor MRI Dataset',
        'num_classes': 4, 'class_names': class_names_ds2,
        'train_loader': train_loader_ds2, 'val_loader': val_loader_ds2, 'test_loader': test_loader_ds2,
    },
    {
        'name': 'Dataset 3 — Brain Tumor MRI Images 44 Classes',
        'num_classes': 44, 'class_names': class_names_ds3,
        'train_loader': train_loader_ds3, 'val_loader': val_loader_ds3, 'test_loader': test_loader_ds3,
    },
]
print('Dataset config ready')

Dataset config ready


## 4. Model  ResNet-34


In [15]:
def build_resnet34(num_classes: int) -> nn.Module:
    """
    ResNet-34 trained from scratch.
    Final FC layer is replaced to match num_classes (4 or 44).

    Architecture summary:
      - Conv1       : 7x7, stride 2
      - Layer1      : 3 x BasicBlock (64 filters)
      - Layer2      : 4 x BasicBlock (128 filters)
      - Layer3      : 6 x BasicBlock (256 filters)
      - Layer4      : 3 x BasicBlock (512 filters)
      - AvgPool + FC: 512 → num_classes
    """
    model = models.resnet34(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)  # 512 → num_classes
    return model.to(DEVICE)


# Sanity check
_m = build_resnet34(4)
_x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
print(f'Output shape    : {list(_m(_x).shape)}')
print(f'Trainable params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')
del _m, _x

Output shape    : [2, 4]
Trainable params: 21,286,724


## 5. Training & Evaluation Functions

In [16]:
def train_one_epoch(model, loader, criterion, optimizer, num_classes):
    model.train()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=num_classes).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def evaluate(model, loader, criterion, num_classes):
    model.eval()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=num_classes).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss_sum += criterion(out, labels).item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def full_evaluation(model, loader, num_classes, class_names):
    model.eval()
    all_preds, all_labels = [], []
    ms = {
        'acc':  Accuracy( task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'prec': Precision(task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'rec':  Recall(   task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'f1':   F1Score(  task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
    }
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        for m in ms.values(): m.update(preds, labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    results = {k: v.compute().item() for k, v in ms.items()}
    results.update({'all_preds': all_preds, 'all_labels': all_labels})

    print(f"  Accuracy : {results['acc']:.4f}")
    print(f"  Precision: {results['prec']:.4f}")
    print(f"  Recall   : {results['rec']:.4f}")
    print(f"  F1-Score : {results['f1']:.4f}")
    print(classification_report(all_labels, all_preds,
          target_names=(class_names if len(class_names) <= 20 else None), digits=4))
    return results


def train_model(model, cfg, save_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val, best_w = float('inf'), None
    nc = cfg['num_classes']

    for ep in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, cfg['train_loader'], criterion, optimizer, nc)
        vl, va = evaluate(model, cfg['val_loader'], criterion, nc)
        scheduler.step(vl)
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        flag = ''
        if vl < best_val:
            best_val, best_w = vl, copy.deepcopy(model.state_dict())
            torch.save(best_w, save_path)
            flag = ' ✅'
        print(f'Ep [{ep:>2}/{NUM_EPOCHS}] Train {tl:.4f}/{ta:.4f} | Val {vl:.4f}/{va:.4f} | {time.time()-t0:.1f}s{flag}')

    model.load_state_dict(best_w)
    return model, history


print('Training & evaluation functions ready')

Training & evaluation functions ready


## 6. Visualization Functions

In [17]:
def plot_curves(history, name, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'ResNet-34 — {name}', fontweight='bold')
    for ax, key, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
        ax.plot(epochs, history[f'train_{key}'], 'b-o', ms=4, label='Train')
        ax.plot(epochs, history[f'val_{key}'],   'r-o', ms=4, label='Val')
        ax.set_title(title); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)
        if key == 'acc': ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_confusion_matrix(results, class_names, name, save_path):
    cm = confusion_matrix(results['all_labels'], results['all_preds'], normalize='true')
    n  = len(class_names)
    fig, ax = plt.subplots(figsize=(min(n*0.7+2, 20), min(n*0.6+2, 18)))
    sns.heatmap(cm, annot=(n<=15), fmt=('.2f' if n<=15 else ''), cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'Confusion Matrix — {name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right', fontsize=(8 if n>15 else 10))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_grad_cam(model, loader, class_names, name, save_path, n=8):
    model.eval()
    # ResNet-34 target layer: last residual block in layer4
    target_layer = model.layer4[-1]
    cam = GradCAM(model=model, target_layers=[target_layer])
    imgs, labels = next(iter(loader))
    imgs = imgs[:n].to(DEVICE)
    with torch.no_grad():
        preds = model(imgs).argmax(1).cpu()
    mean, std = np.array([0.485,0.456,0.406]), np.array([0.229,0.224,0.225])
    fig, axes = plt.subplots(2, n//2, figsize=(n*2, 8))
    fig.suptitle(f'Grad-CAM — {name}', fontweight='bold')
    for i, ax in enumerate(axes.flatten()):
        gc = cam(input_tensor=imgs[i:i+1], targets=[ClassifierOutputTarget(preds[i].item())])[0]
        img_np = np.clip(std * imgs[i].cpu().numpy().transpose(1,2,0) + mean, 0, 1).astype(np.float32)
        ax.imshow(show_cam_on_image(img_np, gc, use_rgb=True))
        ax.set_title(f'True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}',
                     color=('green' if preds[i]==labels[i] else 'red'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


print('Visualization functions ready')

Visualization functions ready


## 7. Train on All 3 Datasets

In [18]:
all_results = {}

for i, cfg in enumerate(DATASETS, start=1):
    print(f'\n{"="*55}\n  Dataset {i}/3: {cfg["name"]}\n  Classes: {cfg["num_classes"]}\n{"="*55}')

    model = build_resnet34(cfg['num_classes'])
    ckpt  = OUTPUT_DIR / f'resnet34_dataset{i}_best.pth'
    model, history = train_model(model, cfg, ckpt)

    print('\n── Test Results ──')
    results = full_evaluation(model, cfg['test_loader'], cfg['num_classes'], cfg['class_names'])
    all_results[cfg['name']] = results

    print('\n── Plots ──')
    plot_curves(history, cfg['name'],                      OUTPUT_DIR / f'resnet34_dataset{i}_curves.png')
    plot_confusion_matrix(results, cfg['class_names'], cfg['name'], OUTPUT_DIR / f'resnet34_dataset{i}_confusion.png')
    plot_grad_cam(model, cfg['test_loader'], cfg['class_names'], cfg['name'], OUTPUT_DIR / f'resnet34_dataset{i}_gradcam.png')

    del model; torch.cuda.empty_cache()

print('\n All datasets complete!')


  Dataset 1/3: Dataset 1 — Brain Tumor Classification (MRI)
  Classes: 4


TypeError: 'NoneType' object is not iterable

## 8. Results Summary

In [ ]:
rows = [
    {'Dataset': name, 'Accuracy': f"{r['acc']:.4f}", 'Precision': f"{r['prec']:.4f}",
     'Recall': f"{r['rec']:.4f}", 'F1-Score': f"{r['f1']:.4f}"}
    for name, r in all_results.items()
]
df = pd.DataFrame(rows, index=[f'Dataset {i+1}' for i in range(len(rows))])
print(df.to_string())
df.to_csv(OUTPUT_DIR / 'resnet34_results.csv')
print(f'\nSaved to {OUTPUT_DIR}/resnet34_results.csv')